# SEBAL Notebook 5: Stability Correction and Calibrated dT Relation
**Scene:** Landsat 7 (2020-01-19)  
**Region:** Mississippi Delta (EPSG:32615)

This notebook applies atmospheric stability corrections to aerodynamic resistance and updates the anchor-based temperature difference (dT) calibration for the SEBAL workflow over the Mississippi Delta. The workflow computes stability-corrected aerodynamic resistance (rah), saves the corrected raster, and recalibrates the linear dT–Ts relationship using the corrected hot-anchor energy balance. The resulting coefficients (a_corr, b_corr) are used in subsequent notebooks to compute refined sensible heat flux, latent heat flux, evaporative fraction, and evapotranspiration.

## 1. Import Required Python Libraries and Utility Functions

In [1]:
import os
import rasterio
import numpy as np
from Utility import get_file_dataframe
from Utility import load_rasters
from Utility import calculate_NDVI
from Utility import calculate_albedo
from Utility import extract_mask_mean
from Utility import read_raster
from Utility import write_raster
from Utility import check_raster

## 2. Set SEBAL working directory

In [2]:
# move from 00_scripts → project root
os.chdir("..")

print("Working directory:", os.getcwd())

Working directory: G:\Collaborations\Mentee\UF_Anitha Madapakula\Scripts\Python\SEBAL_20200119_MSDelta


## 3.  Define core directories and inputs

In [3]:
# === Core directories and scene settings (used across notebooks) ===
clip_dir = "03_clip_landsat"
# output folder
out_dir = "04_indices"
os.makedirs(out_dir, exist_ok=True)
region = 'MSDelta'
proc_dir = "03_processed_met"
date = '20200119'
hour_str = '16'

## 4. Define paths for required rasters

In [4]:
rn_path   = f"{out_dir}/Rn_{date}_{hour_str}Z_{region}.tif"
g_path    = f"{out_dir}/G_{date}_{hour_str}Z_{region}.tif"
lst_path  = f"{out_dir}/LST_C_{date}_{region}.tif"
ndvi_path = f"{out_dir}/NDVI_{date}_{region}.tif"
alb_path  = f"{out_dir}/ALBEDO_{date}_{region}.tif"

## 5. Sensible Heat Flux (H)
### 5.1 Load input raster

In [5]:
# load net radiation raster
Rn, profile, rn_nodata = read_raster(rn_path)

# load soil heat flux raster
G, _, g_nodata = read_raster(g_path)

# load land surface temperature raster
LstC, _, lst_nodata = read_raster(lst_path)

# load NDVI raster
Ndvi, _, ndvi_nodata = read_raster(ndvi_path)

# load albedo raster
Alb, _, alb_nodata = read_raster(alb_path)

# create physically valid NDVI copy for anchor-pixel selection
Ndvi2 = np.where((Ndvi > -1.0) & (Ndvi < 1.0), Ndvi, np.nan)

# build valid mask
valid = (Rn > -9999) & (G > -9999) & (LstC > -9999) & (Ndvi > -9999)

print("Valid pixels:", int(valid.sum()))

Valid pixels: 15410408


### 5.2 Prepare clean mask for anchor selection

In [6]:
# refined valid mask for anchor selection
anchor_valid = (
    (valid) &
    np.isfinite(Rn) & np.isfinite(G) &
    np.isfinite(LstC) & np.isfinite(Ndvi)
)

print("Anchor-valid pixels:", int(anchor_valid.sum()))

Anchor-valid pixels: 15410408


In SEBAL, the hot and cold pixels are selected to represent the two physical extremes of surface energy balance needed to calibrate sensible heat flux (H). The cold pixel represents well-watered vegetation with strong evapotranspiration, so it must have relatively high NDVI and low land surface temperature (LST). The hot pixel represents dry bare soil or sparse vegetation with minimal evapotranspiration, so it must have low NDVI and high LST, following the standard SEBAL framework (Bastiaanssen et al., 1998). Thresholds such as NDVI < 0.2 or NDVI > 0.35 (for winter scenes) reflect realistic vegetation conditions in the scene rather than fixed rules.

In practice, anchor selection follows a polarized logic for stability. Cold pixels (preferred) include dense or moist vegetation with relatively cool LST and uniform surfaces, while hot pixels (preferred) include dry bare soil with warm LST. Percentile-based filtering is commonly used to avoid outliers and improve robustness in operational applications (Allen et al., 2007).

*References*

Allen, R.G., Tasumi, M., & Trezza, R. (2007). Satellite-based energy balance for mapping evapotranspiration with internalized calibration (METRIC). Journal of Irrigation and Drainage Engineering, 133(4), 380–394.

Bastiaanssen, W.G.M., Menenti, M., Feddes, R.A., & Holtslag, A.A.M. (1998). A remote sensing surface energy balance algorithm for land (SEBAL). Journal of Hydrology, 212–213, 198–212.
### 5.3 Prepare for Cold Pixel Selection

In [7]:
# Restrict cold-pixel search to vegetated surfaces
# and retain the coolest 10% of LST within that vegetation mask.# use cleaned NDVI for anchor selection
ndvi_anchor = Ndvi2

# restrict to dense vegetation
cold_mask = anchor_valid & (ndvi_anchor >= 0.35)

print("Cold vegetation pixels:", int(cold_mask.sum()))

# pick lowest temperature inside vegetation
cold_LST_thr = np.nanpercentile(LstC[cold_mask], 10)

cold_pixels = cold_mask & (LstC <= cold_LST_thr)
print("Cold candidates:", int(cold_pixels.sum()))
print("Cold NDVI:", float(np.nanmean(Ndvi[cold_pixels])))
print("Cold LST:", float(np.nanmean(LstC[cold_pixels])))
print("Cold albedo:", float(np.nanmean(Alb[cold_pixels])))
print("Cold G/Rn:", float(np.nanmean((G / Rn)[cold_pixels])))

Cold vegetation pixels: 5062448
Cold candidates: 508958
Cold NDVI: 0.4347250461578369
Cold LST: 6.665546417236328
Cold albedo: 0.10435400903224945
Cold G/Rn: 0.029126230627298355


### 5.4 Refine cold candidates
Cold anchor pixels were refined using NDVI ≥ 0.45 to ensure stable vegetation-driven energy partitioning. Final diagnostics confirm physically consistent cold-anchor conditions.

In [8]:
cold_pixels2 = cold_pixels & (Ndvi >= 0.45)

print("Refined cold count:", int(cold_pixels2.sum()))
print("Refined cold NDVI:", float(np.nanmean(Ndvi[cold_pixels2])))
print("Refined cold LST:", float(np.nanmean(LstC[cold_pixels2])))
print("Refined cold albedo:", float(np.nanmean(Alb[cold_pixels2])))
print("Refined cold G/Rn:", float(np.nanmean((G / Rn)[cold_pixels2])))

Refined cold count: 144856
Refined cold NDVI: 0.5349422693252563
Refined cold LST: 6.642939567565918
Refined cold albedo: 0.11817776411771774
Refined cold G/Rn: 0.028147388249635696


### 5.5 Final cold tightening (temperature only)
Final cold-anchor pixels were defined as the coolest 5% of refined vegetated candidates. Diagnostics confirm physically consistent cold-anchor conditions suitable for SEBAL calibration.

In [9]:
# Final cold-anchor pool:
# dense vegetation with low surface temperature, low albedo,
# and very low G/Rn, consistent with a well-watered cold pixel.
cold_LST_thr2 = np.nanpercentile(LstC[cold_pixels2], 5)
cold_pixels_final = cold_pixels2 & (LstC <= cold_LST_thr2)

print("Final cold count:", int(cold_pixels_final.sum()))
print("Final cold NDVI:", float(np.nanmean(Ndvi[cold_pixels_final])))
print("Final cold LST:", float(np.nanmean(LstC[cold_pixels_final])))
print("Final cold albedo:", float(np.nanmean(Alb[cold_pixels_final])))
print("Final cold G/Rn:", float(np.nanmean((G / Rn)[cold_pixels_final])))

Final cold count: 7277
Final cold NDVI: 0.6127474308013916
Final cold LST: 5.2372870445251465
Final cold albedo: 0.09235788881778717
Final cold G/Rn: 0.01937575824558735


### 5.6 Select Hot Pixel

In [10]:
# use cleaned NDVI for anchor selection
ndvi_anchor = Ndvi2

# restrict to sparse vegetation / bare soil
hot_mask = anchor_valid & (ndvi_anchor <= 0.2)

print("Hot dry pixels:", int(hot_mask.sum()))

# pick hottest part of dry pixels
hot_LST_thr = np.nanpercentile(LstC[hot_mask], 90)
# Final hot-anchor pool:
# sparse vegetation/bare soil with highest LST,
# low NDVI, moderate G/Rn, consistent with dry hot surfaces.
hot_pixels = hot_mask & (LstC >= hot_LST_thr)
hot_pixels2 = hot_pixels & (Ndvi <= 0.15)
hot_LST_thr2 = np.nanpercentile(LstC[hot_pixels2], 95)
hot_pixels_final = hot_pixels2 & (LstC >= hot_LST_thr2)
hot_pixels_final_anchor = hot_pixels_final
print("Hot candidates:", int(hot_pixels.sum()))
print("Hot NDVI:", float(np.nanmean(ndvi_anchor[hot_pixels])))
print("Hot LST:", float(np.nanmean(LstC[hot_pixels])))
print("Hot albedo:", float(np.nanmean(Alb[hot_pixels])))
print("Hot G/Rn:", float(np.nanmean((G / Rn)[hot_pixels])))
print("Final hot count:", int(hot_pixels_final_anchor.sum()))
print("Final hot NDVI:", float(np.nanmean(Ndvi[hot_pixels_final_anchor])))
print("Final hot LST:", float(np.nanmean(LstC[hot_pixels_final_anchor])))
print("Final hot albedo:", float(np.nanmean(Alb[hot_pixels_final_anchor])))
print("Final hot G/Rn:", float(np.nanmean((G / Rn)[hot_pixels_final_anchor])))

Hot dry pixels: 4060022
Hot candidates: 406775
Hot NDVI: 0.09316925704479218
Hot LST: 11.396666526794434
Hot albedo: 0.07418553531169891
Hot G/Rn: 0.049556441605091095
Final hot count: 16034
Final hot NDVI: 0.05736473947763443
Final hot LST: 12.77480697631836
Final hot albedo: 0.0827169269323349
Final hot G/Rn: 0.056394632905721664


### 5.7 Save hot/cold masks

In [11]:
# convert masks to raster form
hot_mask_raster = np.full(Rn.shape, 0, dtype="int16")
cold_mask_raster = np.full(Rn.shape, 0, dtype="int16")

hot_mask_raster[hot_pixels_final] = 1
cold_mask_raster[cold_pixels_final] = 1

# save hot mask
hot_out = f"{out_dir}/hot_pixels_{date}_{hour_str}Z_{region}.tif"
write_raster(raster_path=hot_out, profile=profile, value=hot_mask_raster)

# save cold mask
cold_out = f"{out_dir}/cold_pixels_{date}_{hour_str}Z_{region}.tif"
write_raster(raster_path=cold_out, profile=profile, value=cold_mask_raster)

print("Hot mask sum:", hot_mask_raster.sum())
print("Cold mask sum:", cold_mask_raster.sum())

Saved: 04_indices/hot_pixels_20200119_16Z_MSDelta.tif
Saved: 04_indices/cold_pixels_20200119_16Z_MSDelta.tif
Hot mask sum: 16034
Cold mask sum: 7277


## 6. dT Calibration
### 6.1 Extract masks

In [12]:
# example (edit names as needed)
hot_mask  = os.path.join(out_dir, f"hot_pixels_{date}_{hour_str}Z_{region}.tif")
cold_mask  = os.path.join(out_dir, f"cold_pixels_{date}_{hour_str}Z_{region}.tif")

rasters = {
    "LST":  f"{out_dir}/LST_{date}_{region}.tif",
    "Rn": f"{out_dir}/Rn_{date}_{hour_str}Z_{region}.tif",
    "G": f"{out_dir}/G_{date}_{hour_str}Z_{region}.tif",
    "NDVI": f"{out_dir}/NDVI_{date}_{region}.tif",
    "Albedo": f"{out_dir}/ALBEDO_{date}_{region}.tif"
}

# Dictionary to store anchor values
anchors = {}

for name, path in rasters.items():
    hot_val  = extract_mask_mean(path, hot_mask)    # mean over hot mask
    cold_val = extract_mask_mean(path, cold_mask)   # mean over cold mask

    # store values for later steps
    anchors[name] = {"hot": hot_val, "cold": cold_val}

    # keep print for verification
    print(name, "| HOT:", hot_val, "| COLD:", cold_val)

LST | HOT: 12.774807 | COLD: 5.237287
Rn | HOT: 337.96582 | COLD: 338.01315
G | HOT: 18.994051 | COLD: 6.5118184
NDVI | HOT: 0.05736474 | COLD: 0.61274743
Albedo | HOT: 0.08271693 | COLD: 0.09235789


### 6.2 Compute H at hot and cold pixels
We now compute sensible heat flux H at anchors:
$$
𝐻 = 𝑅_𝑛 − 𝐺 − 𝐿𝐸
$$

In SEBAL anchor logic:

Cold pixel, assume H ≈ 0

Hot pixel, assume LE ≈ 0, so:
$$
𝐻_{ℎot} = 𝑅_{𝑛,ℎ𝑜𝑡} − 𝐺_{ho𝑡}
$$

In [13]:
# 11.2 Convert LST to Kelvin for calibration
Ts_hot  = anchors["LST"]["hot"] + 273.15
Ts_cold = anchors["LST"]["cold"] + 273.15

# Net radiation at hot pixel (W/m²) from anchor extraction
Rn_hot = anchors["Rn"]["hot"] 

# Soil heat flux at hot pixel (W/m²)
G_hot  = anchors["G"]["hot"]

# Sensible heat flux at hot pixel (assume LE ≈ 0 for hot pixel)
H_hot = Rn_hot - G_hot  

# Sensible heat flux at cold pixel (assume H ≈ 0 for cold pixel)
H_cold = 0.0  

# Print results for verification before moving to dT calibration
print("Ts_hot:", Ts_hot)
print("Ts_cold:", Ts_cold)
print("H_hot:", H_hot)

Ts_hot: 285.92480697631834
Ts_cold: 278.3872870445251
H_hot: 318.97177


### 6.3 Solve linear dT relation

In SEBAL
$$
dT = a T_b + b
$$
where, 
$$
b = \frac{dT_{hot} - dT_{cold}}{T_{s,hot} - T_{s,cold}}
$$
$$
b = dT_{hot} - a\,T_{s,hot}
$$

For SEBAL anchor calibration:

- Cold pixel: $dT_{cold} = 0$
- Hot pixel: $dT_{hot}$ will be computed from the anchor energy balance

In [14]:
rah_hot = 110.0          # s/m (neutral initial assumption)
rho_cp = 1.25 * 1004     # J/m³/K

dT_hot = H_hot * rah_hot / rho_cp
dT_cold = 0.0

# correct order
b = (dT_hot - dT_cold) / (Ts_hot - Ts_cold)
a = dT_hot - b * Ts_hot

print("dT_hot:", dT_hot)
print("a:", a)
print("b:", b)

dT_hot: 27.95768512862612
a: -1032.5762565713924
b: 3.709135814115831
